In [1]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

True

In [2]:
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

# RAG
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Loader
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import CSVLoader, WebBaseLoader
from langchain_community.document_loaders import DirectoryLoader


# vector store
from langchain_chroma import Chroma
from langchain_community.vectorstores import FAISS

# Embeddings
from langchain_ollama import OllamaEmbeddings
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_openai import OpenAIEmbeddings

# parser
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser

from langchain_core.prompts import ChatPromptTemplate
from langchain.chat_models import init_chat_model

import json
from langchain_classic.chains.query_constructor.base import AttributeInfo

# selfquery
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever
from langchain_classic.retrievers.self_query.chroma import ChromaTranslator

from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor, EmbeddingsFilter, DocumentCompressorPipeline, CrossEncoderReranker
from langchain_classic.retrievers import EnsembleRetriever, ContextualCompressionRetriever
from langchain_core.structured_query import Comparison, Comparator

# Reranking
from langchain_cohere import CohereRerank

/var/folders/v8/j43szgbd7dl3yp63myxcdyr00000gn/T/ipykernel_2431/817451159.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [4]:
# 모델 선언

llm = init_chat_model(model="gpt-6-luna", reasoning={"effort":"none"})
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

#### 기본 RAG 방식 문제점
- 중복 문서 문제
- 키워드 의존성
- 구조적 질의 불가 : ex) 2023년 이후 계약 중 금액이 1억 이상인 제품
- 노이즈 청크 : 관련성이 낮은 청크가 LLM에게 전달되어 환각 유발
- 긴 컨텍스트 문제 : 불필요한 정보가 많아질수록 LLM 집중도 저하

In [4]:
### 1. Chunking 전략 - chunk_size, chunk_overlap, 임베딩 모델

#### 2. MMR (Maximal Marginal Relevance) Retriever
- 기본 Similarity Search에서 가장 유사한 문서 반환 => 유사한 내용의 문서가 나옴
- 관련성 + 다양성
- fetch_k = 20 추출 => 이미 고른 문서와 덜 비슷한 것 K개 찾기

In [6]:
def create_chunks_from_pdf(pdf_path):
    loader = PyPDFLoader(pdf_path)
    docs = loader.load()
    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap = 50)
    chunks = splitter.split_documents(docs)

    # 빈 chunks 제거 후 반환
    chunks = [chunk for chunk in chunks if chunk.page_content.strip()]
    return chunks


def create_vectorstore(chunks, embedding, collection_name, persist_directory="./db/chroma_db"):
    # ids를 정해서 넘기면 다시 실행해도 같은 id는 덮어써짐 (같은 청크가 계속 쌓이지 않음)
    return  Chroma.from_documents(documents=chunks, 
                                        embedding=embedding, 
                                        persist_directory=persist_directory,
                                        collection_name=collection_name,
                                        ids=[f"{collection_name}-{i}" for i in range(len(chunks))]
                                        )

# 다양성 값 : 0 ~ 1 (일반적인 값 : 0.5)
def create_retriever(vector_store, search_type="similarity", k=4, fetch_k=20, lambda_mult=0.5):

    kwargs = {"k":k}

    if search_type == "mmr":
        kwargs['fetch_k'] = fetch_k
        kwargs['lambda_mult'] = lambda_mult

    return vector_store.as_retriever(search_type=search_type, search_kwargs=kwargs)


def print_retrieved_docs(title, retriever, query):
    docs = retriever.invoke(query)

    print("\n"+"="*50)
    print(title)
    print("="*50)

    for i, doc in enumerate(docs):
        print(f"page={doc.metadata.get("page")}")
        print(doc.page_content)

In [7]:
chunks = create_chunks_from_pdf("./data/제주관광가이드.pdf")
vectorstore = create_vectorstore(chunks=chunks, embedding=embeddings, collection_name="jeju1")

# mmr retriever 생성
mmr_retriever = create_retriever(vector_store=vectorstore, search_type="mmr", k=3, fetch_k=10)

# 기존
sim_retriever = create_retriever(vector_store=vectorstore, k=3)

# query
query = "생활 속 제주어에서 엄불랑하다는 무슨 뜻이야?"
print_retrieved_docs("MMR SEARCH RESULT", mmr_retriever, query)
print_retrieved_docs("SIMILARITY SEARCH RESULT", sim_retriever, query)




MMR SEARCH RESULT
page=12
생활 속 제주어
제주는 타 지역보다 한국어의 고형(古形)을 많이 유지하고 있는 동시에 
제주도만의 고유한 어휘나 문법적 특성을 가지고 있다.
다른 지역 사람이 못 알아듣는 제주어
제주어 뜻풀이
솔쩨기 살짝
안네다 드리다
베지근허다 입안에 기름기가 감돌아 맛이 있다.
엄불랑허다 어마어마하다
코시롱허다 고소하다
산도록허다 시원하다  예) 물이 산도록헌 게 좋다.
두령청이 우두망찰
무사 왜
영, 경, 정 이렇게, 그렇게, 저렇게
게메 글쎄
인사말
제주어 뜻풀이
펜안허우꽈? 편안(안녕)하십니까?
제주도 오난 어떵허우꽈? 제주도에 오니 어떠십니까?
차말로 좋수다. 참말로 좋습니다.
공기도 마고, 산이영 바다잉여 마딱 좋은게마씀 공기도 맑고, 산이랑 바다랑 모두 좋네요.
서울 갈 때랑 하영 다앙 갑서. 서울 갈 때는 많이 담아서 가십시오.
게메양. 경 헤시민 얼마나 좋코마씀? 글쎄요. 그렇게 했으면 얼마나 좋겠습니까?
식당에서
제주어 뜻풀이
무신걸 먹으코? 무엇을 먹을까?
page=13
갈비찜, 생구이, 불고기 등의 다양한 요리로 
응용된다. 말고기는 소고기보다 담백하다.
성게미역국
성게를 넣고 끓인 성게미역국은 제주도 경조사에 올라오는 단골 음식이다. 순두부처럼 엉겨 
달짝지근하면서도 담백한 맛이 난다. 제주에서는 성게를 ‘구살’이라 불러 구살국이라고도 한다.
24JEJU   ISL AND   POCKET   BOOK — 25
Culture  Jeju
page=13
국수문화거리   제주시 이도이동 일원 -민속자연사박물관(삼성로 40) 검색┃제주시 이도동에는 국수를 
좋아하는 제주 사람들은 물론 관광객들을 불러 모으는 국수문화거리가 있어 밤낮으로 푸짐한 국수를 파는 
국숫집들이 사람들을 불러 모으고 있다.
갈치요리                                                                                                                       

#### 3. SelfQuery Retriever
- 자연어 질의를 분석하여 `시멘틱 검색 쿼리` 와 `메타 데이터 필터` 를 LLM이 자동으로 생성해줌

In [7]:
docs = [
    Document(
        page_content="삼성전자 제품 마케팅 직무입니다.",
        metadata = {'year':2025, 'department':'marketing'}
    ), 
    Document(
        page_content="AI 연구 개발 직무입니다.",
        metadata = {'year':2023, 'department':'AI'}
    ), 
    Document(
        page_content="백엔드 개발 직무입니다.",
        metadata = {'year':2025, 'department':'developer'}
    )
]

metadata_field_info = [
    AttributeInfo(
        name='year',
        description='문서 작성 연도',
        type='integer'
    ),
    AttributeInfo(
        name='department',
        description='직무 부서',
        type='string'
    ),
]

document_content_description = "회사 내부 문서 및 정책 자료"

In [8]:
# ChromaTranslator() => 2024년 이후, 이전, => eq, gt, le 로 변환 => LLM 모델에 따라 충분할 수도 있고 아닐수도 잇음

class SafeChromaTranslator(ChromaTranslator):
    def visit_comparison(self, comparison:Comparison):
        range_ops = (Comparator.GT, Comparator.GTE, Comparator.LT,Comparator.LTE)
        v = comparison.value

        if comparison.comparator in range_ops and isinstance(v, str):
            try:
                comparison.value = int(v)
            except ValueError:
                comparison.value = float(v)
        return super().visit_comparison(comparison)

vectorstore = create_vectorstore(chunks=docs, embedding=embeddings, collection_name="self_query1")

selfquery_retriever = SelfQueryRetriever.from_llm(
    llm=llm,
    vectorstore=vectorstore, 
    document_contents=document_content_description, 
    metadata_field_info=metadata_field_info,
    enable_limit=True,
    structured_query_translator=SafeChromaTranslator()
)

In [22]:
query = "2024년 이후 developer 부서 직무 찾아줘"

docs = selfquery_retriever.invoke(query)

for doc in docs:
    print(doc.page_content, doc.metadata)

백엔드 개발 직무입니다. {'year': 2025, 'department': 'developer'}


In [10]:
# 화장품 상품의 설명과 메타데이터 생성
docs = [
    Document(
        page_content="수분 가득한 히알루론산 세럼으로 피부 속 깊은 곳까지 수분을 공급합니다.",
        metadata={"year": 2024, "category": "스킨케어", "user_rating": 4},
    ),
    Document(
        page_content="24시간 지속되는 매트한 피니시의 파운데이션, 모공을 커버하고 자연스러운 피부 표현이 가능합니다.",
        metadata={"year": 2023, "category": "메이크업", "user_rating": 3},
    ),
    Document(
        page_content="식물성 성분으로 만든 저자극 클렌징 오일, 메이크업과 노폐물을 부드럽게 제거합니다.",
        metadata={"year": 2023, "category": "클렌징", "user_rating": 5},
    ),
    Document(
        page_content="비타민 C 함유 브라이트닝 크림, 칙칙한 피부톤을 환하게 밝혀줍니다.",
        metadata={"year": 2023, "category": "스킨케어", "user_rating": 2},
    ),
    Document(
        page_content="롱래스팅 립스틱, 선명한 발색과 촉촉한 사용감으로 하루종일 편안하게 사용 가능합니다.",
        metadata={"year": 2024, "category": "메이크업", "user_rating": 4},
    ),
    Document(
        page_content="자외선 차단 기능이 있는 톤업 선크림, SPF50+/PA++++ 높은 자외선 차단 지수로 피부를 보호합니다.",
        metadata={"year": 2025, "category": "선케어", "user_rating": 5},
    )
]


# 메타데이터 필드 정보 생성
metadata_field_info = [
    AttributeInfo(
        name="category",
        description="The category of the cosmetic product. One of ['스킨케어', '메이크업', '클렌징', '선케어']",
        type="string",
    ),
    AttributeInfo(
        name="year",
        description="The year the cosmetic product was released",
        type="integer",
    ),
    AttributeInfo(
        name="user_rating",
        description="A user rating for the cosmetic product, ranging from 1 to 5",
        type="integer",
    ),
]

document_contents="Brief summary of a cosmetic product"


In [11]:
vectorstore = create_vectorstore(chunks=docs, embedding=embeddings, collection_name="self_query2")

selfquery_retriever2 = SelfQueryRetriever.from_llm(
    llm=llm,
    vectorstore=vectorstore, 
    document_contents=document_content_description, 
    metadata_field_info=metadata_field_info,
    verbose=True,  # 디버깅용
    enable_limit=True,     # 'n개 알려줘' 이런 형태 지원 여부
    structured_query_translator=SafeChromaTranslator()
)

In [12]:
selfquery_retriever2.invoke("2024년 이후로 평점이 4 이상인 제품을 추천해줘")

[Document(id='self_query2-5', metadata={'category': '선케어', 'year': 2025, 'user_rating': 5}, page_content='자외선 차단 기능이 있는 톤업 선크림, SPF50+/PA++++ 높은 자외선 차단 지수로 피부를 보호합니다.')]

In [13]:
selfquery_retriever2.invoke("평점이 4 이상인 제품을 추천해줘")

[Document(id='self_query2-4', metadata={'year': 2024, 'user_rating': 4, 'category': '메이크업'}, page_content='롱래스팅 립스틱, 선명한 발색과 촉촉한 사용감으로 하루종일 편안하게 사용 가능합니다.'),
 Document(id='self_query2-0', metadata={'user_rating': 4, 'category': '스킨케어', 'year': 2024}, page_content='수분 가득한 히알루론산 세럼으로 피부 속 깊은 곳까지 수분을 공급합니다.'),
 Document(id='self_query2-5', metadata={'year': 2025, 'category': '선케어', 'user_rating': 5}, page_content='자외선 차단 기능이 있는 톤업 선크림, SPF50+/PA++++ 높은 자외선 차단 지수로 피부를 보호합니다.'),
 Document(id='self_query2-2', metadata={'year': 2023, 'category': '클렌징', 'user_rating': 5}, page_content='식물성 성분으로 만든 저자극 클렌징 오일, 메이크업과 노폐물을 부드럽게 제거합니다.')]

In [14]:
selfquery_retriever2.invoke("카테고리가 선케어인 상품을 추천해줘")

[Document(id='self_query2-5', metadata={'year': 2025, 'user_rating': 5, 'category': '선케어'}, page_content='자외선 차단 기능이 있는 톤업 선크림, SPF50+/PA++++ 높은 자외선 차단 지수로 피부를 보호합니다.')]

#### 4. BM25 Hybrid Retriever
- 유사도 기반 => 정확한 제품명, 코드, 버전, 고유명사 등의 키워드 매칭에는 약함
- 유사도 + 키워드(TF-IDF) 검색 알고리즘

In [5]:
# json 읽어오기 => Document => 분할하기

from langchain_core.documents import Document

with open("./data/samsung_dx_orc.json", "r", encoding = "utf-8")as f:
    pages = json.load(f)

docs = []

for page in pages:
    docs.append(
        Document(
            page_content=page['content'],
            metadata = {"page":page['page'], "source":"samsung_dx_ocr"}
        )
    )

splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap=50)
chunks = splitter.split_documents(docs)

# 만든 저장소를 변수에 받아야 함 (안 받으면 아래 vectorstore는 셀 13의 화장품 저장소)
ocr_vectorstore = create_vectorstore(chunks=chunks, embedding=embeddings, collection_name="samsung_dx_ocr")

# 의미검색
sim_retriever = create_retriever(ocr_vectorstore)

# BM25 기본 단어 자르기 = 띄어쓰기(split) => "요건에" ≠ "요건", "마케팅을" ≠ "마케팅" (조사 때문에 매칭 실패)
# 한국어용 : 단어를 2글자씩 겹쳐 자르기 (마케팅을 => 마케, 케팅, 팅을) => 조사가 붙어도 앞부분이 겹쳐서 매칭됨
import re

def korean_bigram(text):
    tokens = []
    for word in re.findall(r"[가-힣A-Za-z0-9+#]+", text):   # 문장부호 제거, 단어만
        if len(word) < 3:
            tokens.append(word)
        else:
            tokens += [word[i:i+2] for i in range(len(word) - 1)]
    return tokens

bm25_retriever = BM25Retriever.from_documents(chunks, preprocess_func=korean_bigram)
bm25_retriever.k = 4

# 가중치 0.3 : 0.7 => BM25가 너무 세서 단어만 많이 겹치는 p.1 채용 공고 표가 끼어듦 => 0.5 : 0.5
hybrid_retriever = EnsembleRetriever(retrievers=[sim_retriever, bm25_retriever], weights=[0.5,0.5])

# EnsembleRetriever는 두 결과(4개 + 4개)를 합친 것을 개수 제한 없이 전부 돌려줌 => sim과 같은 4개만 남기기
hybrid_retriever = hybrid_retriever | (lambda docs: docs[:4])

query = "시스템 소프트웨어 자격 요건에 운영체제 개념도 포함되어 있어?"

print_retrieved_docs(title="Hybrid", retriever=hybrid_retriever, query=query)

print_retrieved_docs(title="sim", retriever=sim_retriever, query=query)


Hybrid
page=7
자격요건 Requirements
컴퓨터 전기 전자, 기계 로봇공학 등 관련 전공올 하신 분 운영체제 기본 개념에 대한 이해도틀 보유하신 분 요구사항울 분석하여 소프트웨어들 구조적으로 설계 및 구현하는 역량울 보유하신 분 다양한 분야의 엔지니어와 적극적으로 소통하여 협업할 수 짓는 역량울 보유하신 분
우대사항 Preferences
CIC++ 기반 시스템 프로그래망 역량울 보유하신 분 Linux 기반 임베디드 시스템 개발 경험올 보유하신 분(Kernel, Device Driver, BSP 등) CAN EtherCAT, UDP 등 하드웨어 통신 프로토록 활용 경험 보유하신 분 ROS2 기반 로보텍스 프로적트 수행 경험올 보유하신 분 Git 기반 협업 및 CICD 환경에서의 개발 경험올 보유하신 분
커리어 비전 Career Vision
page=7
미래로봇추진단(서울 근무) S/W개발 시스템 소프트웨어
포지션 소개 Job Overview
휴머노이드 로봇의 실시간 제어 시스템 및 소프트웨어 플렉품올 개발하는 직무입니다: 운영체제 환경 구성, 디바이스 드라이버  실시간 제어 프레임위크 등 로봇 동작의 핵심 기반이 되는 소프트웨어름 설계 및 개발하다, 하드웨어 설계 조직 및 Al 연구 조직과 긴밀히 현업합니다:
수행업무 Job Details
휴머노이드 로봇의 실시간 제어 프레임위크지 설계하고 개발합니다: 모터; 센서 등 로봇 하드웨어 제어름 위한 디바이스 드라이버 및 하드웨어 추상화 계층올 개발합니다. 로봇 제어용 운영제제(Linux 기반 실시간 OS 등) 환경올 구성하고 시스템 성능올 최적화합니다: 유관 부서와 현업하여 보행 조작 등 Al 기능과 제어 시스템 간 연동 미들웨어름 개발합니다:
자격요건 Requirements
page=10
네트위크사업부(수원 근무) S/W개발 미들웨어 개발
포지션 소개 Job Overview
복잡한 네트위크 서비스의 뼈대름 세우고 시스템 자원올 스마트하게 통제하는 최적의 슬루선올 만드는 직무입니다
수행업무 Job 

#### 5. Reranker
- 후보군 가져와서 질문 - 청크 쌍을 정밀하게 채점
- Bi-Encoder : 쿼리, 문서 독립적으로 처리 (대규모 검색에 우수함)
- Cross-Encoder : 쿼리 + 문서 쌍으로 처리 (느릴 순 있으나 정확도 매우 높음)

- 1차 검색 : Bi-Encoder 처리 => 2차 검색 : Cross-Encoder
- Cross-Encoder => Cohere Reranker

In [ ]:
# 1차 검색 (Bi-Encoder) : 셀 18의 삼성 저장소(64개)에서 후보 20개
# (vectorstore는 셀 13의 화장품 저장소라서 쓰면 안 됨)
base_retriever = create_retriever(ocr_vectorstore, k=20)

# 2차 검색 (Cross-Encoder) : 후보 20개를 질문과 한 쌍씩 다시 채점 => 상위 5개만 남김
rerank = CohereRerank(model="rerank-v4.0-pro", top_n=5)

# 1차 => 2차 순서로 연결 (문서 내용을 자르지 않고 순서만 다시 매김)
compression_retriever = ContextualCompressionRetriever(base_compressor=rerank, base_retriever=base_retriever)

# 문서에 있는 내용으로 질문 (셀 18과 같은 질문 => sim, Hybrid와 비교)
query = "시스템 소프트웨어 자격 요건에 운영체제 개념도 포함되어 있어?"

# print_retrieved_docs(title="Rerank + Sim", retriever=compression_retriever, query=query)


docs = compression_retriever.invoke(query)

for doc in docs:
    print(f"rele_score {doc.metadata.get("relevance_score", "N/A")}")
    print(doc.page_content[:200])
    print("===")


rele_score 0.9281097
자격요건 Requirements
컴퓨터 전기 전자, 기계 로봇공학 등 관련 전공올 하신 분 운영체제 기본 개념에 대한 이해도틀 보유하신 분 요구사항울 분석하여 소프트웨어들 구조적으로 설계 및 구현하는 역량울 보유하신 분 다양한 분야의 엔지니어와 적극적으로 소통하여 협업할 수 짓는 역량울 보유하신 분
우대사항 Preferences
CIC++ 기반 시스템 프로
===
rele_score 0.85126173
미래로봇추진단(서울 근무) S/W개발 시스템 소프트웨어
포지션 소개 Job Overview
휴머노이드 로봇의 실시간 제어 시스템 및 소프트웨어 플렉품올 개발하는 직무입니다: 운영체제 환경 구성, 디바이스 드라이버  실시간 제어 프레임위크 등 로봇 동작의 핵심 기반이 되는 소프트웨어름 설계 및 개발하다, 하드웨어 설계 조직 및 Al 연구 조직과 긴밀히 현업합
===
rele_score 0.658066
자격요건 Requirements
전자공학 통신공학 RF 및 시스템 설계 관련 전공올 하신 분 Python/R 등 Al 활용 프로그래망 능력 및 Al 활용한 테스트 자동화 경험올 보유하신 분 RF-전자기장 신호 전달, 임피터스 매칭, 노이즈 분석에 대한 이해가 높으신 분 PCB 설계 Tool 및 다양한 시물레이선 설계 및 검증 Tool 활용 능력이 뛰어나신 
===
rele_score 0.65277284
네트위크사업부(수원 근무) S/W개발 미들웨어 개발
포지션 소개 Job Overview
복잡한 네트위크 서비스의 뼈대름 세우고 시스템 자원올 스마트하게 통제하는 최적의 슬루선올 만드는 직무입니다
수행업무 Job Details
애플리키이선 개발 및 운용에 필요한 공통 라이브러리블 개발합니다: 가상화 CaaS 지원 K8S API, 메시장 서비스(gRPC Pro
===
rele_score 0.62758607
MX사업부(수원 근무) S/W개발 인공지능
포지션 소개 Job Overview
MX사업부의 다양한 디바이스(Smartphone  W

#### LLMChainExtractor
- 질문 + 문서를 같이 LLM에게 보내서 질문과 관련된 내용만 추출
- 현재 받은 chunks 안에 있는 내용을 줄여내는 것

In [9]:
extractor = LLMChainExtractor.from_llm(llm=llm)

# 기존꺼 호출
vectorstore = Chroma(collection_name="jeju1", persist_directory="./db/chroma_db", embedding_function=embeddings)

base_retriever = create_retriever(vectorstore, k=20)
query = "생활 속 제주어에서 엄불랑하다는 무슨 뜻이야?"

base_retriever.invoke(query)


extractor = LLMChainExtractor.from_llm(llm=llm)
compression_retriever = ContextualCompressionRetriever(base_retriever=base_retriever, base_compressor=extractor)
docs = compression_retriever.invoke(query)

print(len(docs))

for doc in docs:
    print(doc.page_content[:200])
    print("===")

1
엄불랑허다 어마어마하다
===


In [12]:
base_retriever = create_retriever(vectorstore, k=20)

embedding_filter = EmbeddingsFilter(embeddings=embeddings, similarity_threshold=0.5)

# embedding_filter => extractor

pipeline = DocumentCompressorPipeline(transformers=[embedding_filter, extractor])

compression_retriever = ContextualCompressionRetriever(base_compressor=pipeline, base_retriever=base_retriever)

print_retrieved_docs(title="EmbeddingsFilter + LLMChainExtractor", retriever=compression_retriever, query=query)


EmbeddingsFilter + LLMChainExtractor
page=12
엄불랑허다 어마어마하다
